# Imports

In [ ]:
# Para ejecutarlo en Google Colab hay que instalar estas librerías

install.packages("remotes")
install.packages("kableExtra")
devtools::install_github("stan-dev/cmdstanr")
install.packages(c("coda","mvtnorm","devtools","loo","dagitty"))
devtools::install_github("rmcelreath/rethinking")

In [ ]:
# Instalar cmdstan
if (is.null(tryCatch(cmdstan_path(), error = function(e) NULL))) {
  install_cmdstan(cores = 2)
}

In [ ]:
library(rethinking)
library(tidyverse)
library(knitr)
library(kableExtra)
library(cmdstanr)

# Usar cmdstanr como backend
options(mc.cores = parallel::detectCores())
rethinking::set_ulam_cmdstan(TRUE)

out_dir <- "resultados_multinivel"
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)
cat("Guardando resultados en:", out_dir, "\n")

filepath = 'input_regresion.csv'

df <- read.csv(filepath) %>%
  filter(base != "vivo en") # Saco la oración de control

# Limpieza de datos

In [ ]:
df <- df %>%
  mutate(
    # ID numérico por modelo (efecto aleatorio)
    model_id = as.integer(as.factor(model)),
    # ID numérico por par de oraciones (efecto aleatorio)
    pair_id  = as.integer(as.factor(paste(base, pair, sep = "_")))
  )

dat_raw <- df %>%
  select(delta, q_param, it, created_at, corpus_count, token_count_ar,
         model_id, pair_id, family, model, q_param) %>%
  mutate(it = as.integer(it == "True")) %>%
  drop_na()

dat <- dat_raw %>%
  mutate(
    delta_z        = standardize(delta),
    q_param_z      = standardize(q_param),
    created_at_z   = standardize(created_at),
    corpus_count_z = standardize(corpus_count),
    token_count_z  = standardize(token_count_ar)
  )

n_models <- length(unique(dat$model_id))
n_pairs  <- length(unique(dat$pair_id))

dat_list <- list(
  delta_z        = dat$delta_z,
  it             = dat$it,
  q_param_z      = dat$q_param_z,
  created_at_z   = dat$created_at_z,
  corpus_count_z = dat$corpus_count_z,
  token_count_z  = dat$token_count_z,
  model_id       = dat$model_id,
  pair_id        = dat$pair_id
)

# Ajuste

In [ ]:
# Modelo multinivel
#
#  Efectos aleatorios (non-centered):
#    - Por modelo: z_model[model_id] ~ Normal(0,1) → alpha_model = z_model * sigma_model
#    - Por par: z_pair[pair_id]   ~ Normal(0,1) → alpha_pair  = z_pair  * sigma_pair
#
#  Efectos fijos:
#    it, q_param, created_at, corpus_count, token_count
#
#  Heterocedasticidad:
#    log(sigma) = gamma_base + gamma_q_param * q_param + gamma_it * it

ruta_modelo <- file.path(out_dir, "modelo_ulam.rds")

if (file.exists(ruta_modelo)) {

  cat("Modelo en disco\n")
  m <- readRDS(ruta_modelo)

} else {

  cat("Ajustando modelo\n")

  m <- ulam(
    alist(
      # Likelihood
      delta_z ~ dnorm(mu, sigma_i),

      # Media: intercepto global + efectos aleatorios + efectos fijos
      mu <- mu_alpha +
            z_model[model_id] * sigma_model +
            z_pair[pair_id]   * sigma_pair  +
            beta_it           * it           +
            beta_q_param      * q_param_z    +
            beta_created_at   * created_at_z +
            beta_corpus_count * corpus_count_z +
            beta_token_count  * token_count_z,

      # Heterocedasticidad
      log(sigma_i) <- gamma_base +
                      gamma_q_param * q_param_z +
                      gamma_it      * it,

      # Efectos aleatorios (non-centered)
      z_model[model_id] ~ dnorm(0, 1),
      z_pair[pair_id]   ~ dnorm(0, 1),

      # Hiperpriors
      mu_alpha    ~ dnorm(0, 1),
      sigma_model ~ dexp(1),
      sigma_pair  ~ dexp(1),

      # Priors efectos fijos
      beta_it           ~ dnorm(0, 1),
      beta_q_param      ~ dnorm(0, 1),
      beta_created_at   ~ dnorm(0, 1),
      beta_corpus_count ~ dnorm(0, 1),
      beta_token_count  ~ dnorm(0, 1),

      # Priors modelo de sigma
      gamma_base    ~ dnorm(0, 1),
      gamma_q_param ~ dnorm(0, 1),
      gamma_it      ~ dnorm(0, 1)
    ),
    data    = dat_list,
    chains  = 4,
    cores   = 4,
    iter    = 2000,
    warmup  = 1000,
    log_lik = TRUE
  )

  saveRDS(m, ruta_modelo)
  cat("Modelo guardado en:", ruta_modelo, "\n")

}

# Posterior estimada

In [ ]:
post         <- extract.samples(m)
model_labels <- levels(as.factor(dat$model))
pair_labels  <- levels(as.factor(paste(dat_raw$base, dat_raw$pair, sep = "_")))

In [ ]:
p     <- precis(m, depth = 2, prob = 0.89)
p_mat <- as.data.frame(p)          # convertir UNA sola vez acá
p_rn  <- rownames(p_mat)

# Renombrar filas
p_rn <- sapply(p_rn, function(x) {
  if (grepl("z_model\\[\\d+\\]", x)) {
    i <- as.integer(sub(".*\\[(\\d+)\\]", "\\1", x))
    paste0("alpha_model[", model_labels[i], "]")
  } else if (grepl("z_pair\\[\\d+\\]", x)) {
    i <- as.integer(sub(".*\\[(\\d+)\\]", "\\1", x))
    paste0("alpha_pair[", i, "]")
  } else { x }
})

rownames(p_mat) <- p_rn
tab <- p_mat
tab$param <- p_rn
tab <- tab[, c("param", setdiff(names(tab), "param"))]
write.csv(tab, file.path(out_dir, "tabla_posterior.csv"), row.names = FALSE)

# Guardar tabla como LaTeX
kable(
  tab[, !names(tab) %in% "param"],
  format    = "latex",
  booktabs  = TRUE,
  digits    = 2,
  caption   = "Resumen posterior",
  row.names = TRUE
) %>%
  save_kable(file.path(out_dir, "tabla_posterior.tex"))

cat("Tablas guardadas.\n")

# Gráficos

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 7)


p_df <- tab

p_global <- p_df %>%
  filter(!grepl("alpha_model\\[|alpha_pair\\[", param)) %>%
  mutate(param = factor(param, levels = rev(param)))

g1 <- ggplot(p_global, aes(x = mean, y = param)) +
  geom_errorbarh(aes(xmin = `5.5%`, xmax = `94.5%`), height = 0.3) +
  geom_point(size = 2) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  labs(
    x     = "Valor posterior (media e intervalo 89%)",
    y     = NULL
  ) +
  theme_classic(base_size = 13)

ggsave(file.path(out_dir, "plot_parametros_globales.png"),
       g1, width = 10, height = 6, dpi = 150)

# Interceptos por modelo (reconstruidos desde non-centered) ---

alpha_model_matrix <- sapply(1:n_models, function(j) {
  post$mu_alpha + post$z_model[, j] * post$sigma_model
})

alpha_model_df <- data.frame(
  model  = model_labels,
  mean   = colMeans(alpha_model_matrix),
  lo     = apply(alpha_model_matrix, 2, function(x) quantile(x, 0.055)),
  hi     = apply(alpha_model_matrix, 2, function(x) quantile(x, 0.945)),
  family = family_map[model_labels]
) %>%
  arrange(mean) %>%
  mutate(model = factor(model, levels = model))

g2 <- ggplot(alpha_model_df, aes(x = mean, y = model, color = family)) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.3) +
  geom_point(size = 2) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  labs(
    x      = "Alpha posterior (media e intervalo 89%)",
    y      = NULL,
    color  = "Familia"
  ) +
  theme_classic(base_size = 11) +
  theme(axis.text.y = element_text(size = 7))

ggsave(file.path(out_dir, "plot_interceptos_modelo.png"),
       g2, width = 11, height = 9, dpi = 150)

# Delta mediana por q_param, color por family

df_plot <- dat %>%
  group_by(q_param, family) %>%
  summarise(mean_delta = median(delta), .groups = "drop")

g3 <- ggplot(df_plot, aes(x = q_param, y = mean_delta, color = family)) +
  geom_point(size = 2) +
  xlim(0, 10) +
  labs(
    x      = "Cantidad de parámetros (miles de millones)",
    y      = "Delta (mediana)",
    color  = "Familia"
  ) +
  theme_minimal(base_size = 13)

ggsave(file.path(out_dir, "plot_delta_qparam_family.png"),
       g3, width = 10, height = 5, dpi = 150)

#  Distribución de aciertos por par 

df_pairs <- df %>%
  mutate(binary_num = as.integer(binary_delta == "True")) %>%
  group_by(pair, base) %>%
  summarise(aciertos = sum(binary_num, na.rm = TRUE), .groups = "drop") %>%
  count(aciertos, name = "frecuencia")

g4 <- ggplot(df_pairs, aes(x = aciertos, y = frecuencia)) +
  geom_col(fill = "steelblue", color = "white") +
  scale_x_continuous(breaks = unique(df_pairs$aciertos)) +
  labs(
    x     = "Suma de aciertos por par (delta > 0)",
    y     = "Frecuencia"
  ) +
  theme_minimal(base_size = 13)

ggsave(file.path(out_dir, "plot_aciertos_par.png"),
       g4, width = 10, height = 5, dpi = 150)